In [1]:
import imaplib
import email
import os

from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
username = os.getenv("USERNAME")
password = os.getenv("PASSWORD")
download_folder = os.getenv("DOWNLOAD_FOALDER")

imap_server = "imap.strato.de"

In [4]:
mail = imaplib.IMAP4_SSL(imap_server)
mail.login(username, password)
mail.select("INBOX")

('OK', [b'1375'])

In [5]:
status, messages = mail.search(None, "ALL")

mail_ids = messages[0].split()

for mail_id in mail_ids[-2:]:
    status, msg_data = mail.fetch(mail_id, "(RFC822)")

    raw_email = msg_data[0][1]
    msg = email.message_from_bytes(raw_email)

    print("From:", msg["From"])
    print("Subject:", msg["Subject"])

    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition"))

            print(f"Testing {content_disposition} + {content_type}")

            if content_type == "text/plain" and (
                content_disposition is None
                or "attachment" not in content_disposition.lower()
            ):
                print("Text:")
                print(part.get_payload(decode=True).decode(errors="ignore"))

            elif content_type == "text/html":
                print("HTML:")
                print(part.get_payload(decode=True).decode(errors="ignore"))

            elif part.get_filename():
                filename = part.get_filename()
                print("Attachment found:", filename)
                filepath = os.path.join(download_folder, filename)
                with open(filepath, "wb") as f:
                    f.write(part.get_payload(decode=True))

    print("------")

From: "HelloFresh" <news@newsletter.hellofresh.de>
Subject: =?UTF-8?B?8J+llyBHZW5pZcOfZSB1bnNlciBORVVFUyBNZW7DvCArIHNwYXJl?=
 =?UTF-8?B?IGJpcyB6dSA2MCUgYXVmIDQgQm94ZW4h?=
------
From: Academia Mentions <premium@academia-mail.com>
Subject: =?UTF-8?Q?Are_you_the_=E2=80=9CL._Petersdorf=E2=80=9D_mentioned_in?=
 =?UTF-8?Q?_Machine_Learning_papers=3F?=
Testing None + multipart/alternative
Testing None + text/plain
Text:
Dear Lukas,

The name “L. Petersdorf” is mentioned in a Machine Learning paper.

Follow the link below to see all of your mentions:

https://www.academia.edu/keypass/UHY5b2VmbW5tcDJETm5WUk9kOUptWWVNOHZ5aEpReVp6TVNINXFTRkRrdz0tLUVHYkdoOWNmSU5jWnFNZGZ6MnZQeFE9PQ==--936882ef2d1bc7274e2099f50924c351e1a255cd/t/FdwRe-TjAEHFJ-btaxf6/upgrade?after_upgrade_path=https%3A%2F%2Fwww.academia.edu%2Ft%2FFdwRe-TjAEHFJ-btaxf6%2Fmentions%3Ffeatured%3D129745519366%26top_mention_ids%3D129745519366&feature=name_mentions&featured=129745519366&mentions_color=ri%3AMachine+Learning&trigger=new-name-m

In [6]:
mail.logout()

('BYE', [b'Logout'])